# 21 - Demand-Weighted Criticality

Every criticality ranking produced so far in this project is **purely topological**: a station is important because of where it sits in the graph, not because of how many people would actually be left stranded if it failed. This notebook explores the second follow-up direction of the report - *weighting criticality by passenger demand instead of by topology alone* - and asks whether the answer to "which stations are critical" changes once you put people back into the network.

## Read this before reading any number below

**The GTFS file contains no ridership data of any kind.** No boardings, no alightings, no fare validations, no load factors, and no origin-destination matrix in any file of the Israeli feed. So everything this notebook calls *demand* is a **PROXY**, built from two things we do have:

1. **Residential population** by Central Bureau of Statistics 2021 statistical area, joined to stops in notebook `09_socioeconomic_equity`; and
2. **Service volume** (`stop_use_count`, the number of scheduled trip visits at a stop), used only as a mild multiplier on boarding propensity.

The proxy is marked as such everywhere it appears: in the text, in the title of every figure, and in the output column names (`population_served`, `demand_weight`, `demand_weighted_betweenness_proxy`). Section 15 goes through the ways the proxy is wrong in detail. Nothing here should be read as an estimate of actual passenger counts.

## The research question

If you weight shortest-path criticality by the number of residents living at a trip's origin and destination, **which stations gain importance and which lose it** relative to the pure topological ranking of notebook `04_centrality_analysis`? Does a structurally negligible station in a dense neighborhood rank higher than a well-connected station in an empty area?

## The methodology in one paragraph

Each CBS statistical area's population is split among the stops that sit inside it, then reallocated across neighboring stops using a gravity kernel with distance decay (so a resident is shared among the stops actually within walking range, and the national total is preserved). That gives a `population_served` for each stop. Multiplying by the service-volume term yields `demand_weight`. We then run an estimate of **demand-weighted betweenness**: a Brandes algorithm in which source stops are sampled with probability proportional to their demand weight, and each destination stop contributes its demand weight to the dependency accumulation. The result estimates `sum_{s,t} w(s) w(t) sigma_st(v) / sigma_st` - shortest-path criticality where each path counts for the number of people plausibly at its two ends. We compare it against notebook 04's `approx_betweenness`, and against a **uniform-weight control** run through the *same* estimator, so the comparison isolates the effect of demand weighting rather than the effect of swapping the estimator.

## Input (must exist before running this notebook)

* `outputs/nb/02_graph_construction/tables/nodes.csv`, `edges.csv` - the trip-adjacency graph (notebook **02**).
* `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` - the topological baseline, specifically the column `approx_betweenness` (notebook **04**).
* `outputs/nb/09_socioeconomic_equity/tables/stops_with_socioeconomic.csv` and `socioeconomic_neighborhood_access.csv` - the join to CBS statistical areas, the population, and the socioeconomic cluster (notebook **08**).

No raw GTFS file is read here: `stop_times.txt` (816 MB) is **not** needed, because the population proxy is built from stage 09's output and the graph from stage 02's output.

## Output (all under `outputs/nb/21_demand_weighted_criticality/`)

* `tables/demand_proxy.csv` - `stop_id, population_served, demand_weight` (the contract file).
* `tables/demand_weighted_criticality.csv` - `stop_id, stop_name, topological_rank, demand_weighted_rank, rank_shift` (the contract file).
* `demand_summary.json` - the headline numbers, the parameters, and the agreement statistics.
* `tables/demand_proxy_detail.csv` - the proxy with all its intermediate columns and the CBS join quality per stop.
* `tables/criticality_lens_detail.csv` - the three score columns and three rank columns per stop.
* `tables/rank_agreement.csv` - Spearman rho and top-N overlap between the lenses.
* `tables/rank_movers.csv` - the biggest gainers and losers in the ranking.
* `tables/rank_shift_by_socioeconomic_cluster.csv` - mean rank shift per CBS cluster 1-10.
* `tables/proxy_sensitivity.csv`, `tables/betweenness_sensitivity.csv` - how much the proxy parameters matter.
* `figures/*.png`.

Nothing outside this stage's folder is written.

## 1. Initialize the working environment

The cell below is the project's standard bootstrap, identical to the one in notebook 03, so this notebook runs both on a local checkout and on Google Colab. `_ensure(...)` pip-installs only the packages that are genuinely missing (a repeat run is cheap), and `find_repo_root()` climbs upward from the working directory looking for the GTFS folder, cloning the repository into `/content` if it isn't found (the Colab case). It sets `REPO`, `DATA`, and `OUT`, which every later cell depends on, so it has to run first.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, stage folders, and tunable constants

This stage owns exactly one output folder, `outputs/nb/21_demand_weighted_criticality/`, with the subfolders `tables/` and `figures/`, following the project convention.

All the runtime and all the modeling choices are gathered in the constants below, so a reviewer can change the model or trade runtime in exactly one place:

* `CATCHMENT_RADIUS_M = 500` - a stop's walking catchment, in meters. Residents farther than this aren't assigned to it. 500 m is the standard catchment for a bus stop; 800 m is often used for rail.
* `DECAY_LENGTH_M = 250` - the gravity kernel is `exp(-d / DECAY_LENGTH_M)`, so a stop 250 m away gets `1/e` of the pull of a stop right at the doorstep.
* `SERVICE_EXPONENT = 0.5` - how strongly service volume multiplies the population term (`0` = population only, `1` = fully proportional to scheduled visits). The sensitivity run in section 14 shows what this choice costs.
* `K_DEMAND_SOURCES = 400` - the number of sampled sources per betweenness lens. **This is the main cost knob.** Each source is one BFS plus one dependency accumulation over the largest component of ~30k nodes, about 0.2-0.5 s in pure Python, and we run **two** lenses (demand-weighted and the uniform control), so expect about 3-8 minutes here. You can lower it to 150 for a quick pass; the estimate is unbiased at any `k`, just noisier.
* `RUN_WEIGHT_SENSITIVITY` / `RUN_BETWEENNESS_SENSITIVITY` - the two sensitivity blocks. The first is cheap (a few seconds per catchment radius). The second reruns the betweenness estimator with `SENSITIVITY_K = 150` sources for two alternative proxy definitions, adding a few more minutes; you can set it to `False` if you're in a hurry, but section 14 is part of the methodological-integrity argument.

The betweenness estimator is written out in full in section 9 and isn't taken from `networkx`, because `networkx.betweenness_centrality` can't weight the endpoints - and that's the entire point of this notebook.

In [ ]:
# --- Libraries, stage folders and every tunable constant -------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn', 'scipy')

import json
import math
import time
from collections import deque

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

STAGE = OUT / '21_demand_weighted_criticality'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Demand PROXY parameters ----------------------------------------------
CATCHMENT_RADIUS_M = 500.0    # walking catchment of a stop, metres
DECAY_LENGTH_M = 250.0        # gravity kernel exp(-d / DECAY_LENGTH_M)
SERVICE_EXPONENT = 0.5        # 0 = population only, 1 = fully service-proportional

# --- Betweenness estimator cost -------------------------------------------
K_DEMAND_SOURCES = 400        # sampled sources per lens (main cost knob)
SEED_DEMAND = 21              # seed of the demand-weighted lens
SEED_UNIFORM = 22             # seed of the uniform-weight control lens
PROGRESS_EVERY = 50           # print a progress line every N sampled sources

# --- Sensitivity blocks ----------------------------------------------------
RUN_WEIGHT_SENSITIVITY = True        # cheap: re-computes the proxy only
RUN_BETWEENNESS_SENSITIVITY = True   # costly: two extra estimator runs
SENSITIVITY_K = 150                  # sources used inside the sensitivity runs
RADIUS_GRID = [300.0, 500.0, 1000.0]
SERVICE_EXPONENT_GRID = [0.0, 0.5, 1.0]

# --- Presentation ----------------------------------------------------------
TOP_N = 15                    # rows in the top-N tables / bars in the charts
MOVERS_N = 25                 # gainers and losers written to rank_movers.csv
OVERLAP_SIZES = [50, 200, 1000]
FIG_DPI = 150

print('stage folder :', STAGE)
print('networkx', nx.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)

## 3. Rendering Hebrew labels

Station names in the Israeli GTFS feed are in Hebrew, and several figures below print them (in particular the rank-shift charts - the whole point is *which* stations move). Matplotlib doesn't implement the Unicode bidirectional algorithm, so right-to-left text renders reversed. The cell below is the project's standard one-time monkey-patch of `matplotlib.text.Text.set_text`: any string containing Hebrew characters is converted to display order using `python-bidi` before drawing, and a font with Hebrew glyphs is selected (Arial on Windows, DejaVu Sans elsewhere). The operation is idempotent. All the other text in this notebook is in English.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Locating the earlier stages

This notebook consumes three earlier stages. Instead of hard-coding folder names, `stage_dir` locates a stage by its **two-digit prefix** (`OUT.glob('04*')`), so it keeps working even if a stage folder is renamed from `04_centrality_analysis` to anything else starting with `04`. Then `load_stage_table` searches that folder recursively for the file, so it doesn't matter whether the table sits at the folder root or inside `tables/`.

Both helpers raise a `FileNotFoundError` that says **which notebook to run first**, because a missing artifact from an earlier stage is the most likely reason this notebook fails for someone else.

In [ ]:
# --- Resolve earlier stages by two-digit prefix ----------------------------
def stage_dir(prefix, notebook_hint):
    """Return the stage folder whose name starts with `prefix`, else raise."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            f'No stage folder starting with "{prefix}" under {OUT} - '
            f'run notebook {notebook_hint} first.'
        )
    return matches[0]


def load_stage_table(prefix, filename, notebook_hint, **read_kwargs):
    """Load a CSV written by an earlier stage, with an actionable error message."""
    folder = stage_dir(prefix, notebook_hint)
    candidates = [folder / 'tables' / filename, folder / filename]
    path = next((c for c in candidates if c.exists()), None)
    if path is None:
        found = sorted(folder.rglob(filename))
        path = found[0] if found else None
    if path is None:
        raise FileNotFoundError(
            f'"{filename}" not found anywhere under {folder} - '
            f'run notebook {notebook_hint} first; it is the notebook that writes it.'
        )
    frame = pd.read_csv(path, encoding='utf-8-sig', **read_kwargs)
    print(f'{filename:<42} {len(frame):>7,} rows  <-  {path}')
    return frame


def require_columns(frame, columns, filename):
    """Fail loudly if an upstream contract column is missing."""
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(
            f'{filename} is missing the column(s) {missing}. '
            f'It has {list(frame.columns)}. Re-run the notebook that produces it.'
        )


print('helpers ready')

## 5. Loading the input

Five tables are read:

* `nodes.csv` / `edges.csv` (stage 02) - the trip-adjacency graph and the stop attributes, including `stop_use_count`, the number of scheduled trip visits at each stop, which is our service-volume term.
* `stop_metrics.csv` (stage 04) - the topological baseline. The column we compare against is `approx_betweenness`; note that it is itself a `k`-sampled estimate (notebook 04 used 300 sources), so it carries its own sampling noise into every comparison below.
* `stops_with_socioeconomic.csv` (stage 09) - the CBS statistical area for each stop (`socio_unit_id`), the area's population (`socio_population`), its socioeconomic cluster 1-10 (`socio_cluster`), and importantly, **how the join was done** (`socio_join_method` = `within` for point-in-polygon matching, `nearest` for a fallback to the closest polygon, plus `socio_join_distance_m`).
* `socioeconomic_neighborhood_access.csv` (stage 09) - the same areas in aggregate form, used to fill in an area population missing from the stop-level table and to cross-check the number of stops per area.

We validate the contract columns right away, so a schema change in an earlier stage fails here instead of quietly producing a wrong ranking.

In [ ]:
# --- Load the artifacts of notebooks 02, 04 and 08 -------------------------
nodes_df = load_stage_table('02', 'nodes.csv', '02_graph_construction',
                            dtype={'stop_id': str})
edges_df = load_stage_table('02', 'edges.csv', '02_graph_construction',
                            dtype={'from_stop': str, 'to_stop': str})
metrics_df = load_stage_table('04', 'stop_metrics.csv', '04_centrality_analysis',
                              dtype={'stop_id': str})
socio_df = load_stage_table('08', 'stops_with_socioeconomic.csv',
                            '09_socioeconomic_equity', dtype={'stop_id': str})
access_df = load_stage_table('08', 'socioeconomic_neighborhood_access.csv',
                             '09_socioeconomic_equity')

require_columns(nodes_df, ['stop_id', 'stop_name', 'lat', 'lon', 'region', 'stop_use_count'],
                'nodes.csv')
require_columns(edges_df, ['from_stop', 'to_stop', 'trip_frequency'], 'edges.csv')
require_columns(metrics_df, ['stop_id', 'approx_betweenness'], 'stop_metrics.csv')
require_columns(socio_df, ['stop_id', 'socio_unit_id', 'socio_population', 'socio_cluster'],
                'stops_with_socioeconomic.csv')
require_columns(access_df, ['socio_unit_id', 'population', 'stops'],
                'socioeconomic_neighborhood_access.csv')

nodes_df['stop_id'] = nodes_df['stop_id'].astype(str)
metrics_df['stop_id'] = metrics_df['stop_id'].astype(str)
socio_df['stop_id'] = socio_df['stop_id'].astype(str)

print()
print(f'stops in the graph          : {nodes_df["stop_id"].nunique():,}')
print(f'stops with a CBS join       : {socio_df["socio_unit_id"].notna().sum():,}')
print(f'CBS statistical areas        : {access_df["socio_unit_id"].nunique():,}')
print(f'stops covered by stage 04    : {metrics_df["stop_id"].nunique():,}')

## 6. Rebuilding the undirected graph and its largest component

Criticality here is a shortest-path quantity, so, as everywhere else in the project, it is computed on the **undirected projection** `G` of the trip graph (one edge per unordered stop pair, with the trip frequencies of the two directions summed) and restricted to the **largest connected component** `Gc`. Stops outside the largest component can't lie on a shortest path within it, so their score is 0 in every lens - that's the honest value, not a missing value.

Shortest paths are counted in **hops** (the minimum number of segments), exactly as in notebook 04, so the topological baseline and the demand-weighted lens differ *only* in how the endpoints are weighted. Travel-time-weighted paths would be a different (and legitimate) modeling choice; mixing both changes at once would make the comparison uninterpretable.

In [ ]:
# --- Undirected projection and largest connected component -----------------
def build_undirected_graph(nodes_frame, edges_frame):
    """One edge per unordered station pair; weights of both directions summed."""
    graph = nx.Graph()
    graph.add_nodes_from(nodes_frame['stop_id'].astype(str))
    freq = pd.to_numeric(edges_frame['trip_frequency'], errors='coerce').fillna(1.0)
    for u, v, w in zip(edges_frame['from_stop'].astype(str),
                       edges_frame['to_stop'].astype(str),
                       freq.astype(float)):
        if u == v:
            continue
        if graph.has_edge(u, v):
            graph[u][v]['weight'] += w
        else:
            graph.add_edge(u, v, weight=w)
    return graph


G = build_undirected_graph(nodes_df, edges_df)
components = sorted(nx.connected_components(G), key=len, reverse=True)
Gc = G.subgraph(components[0]).copy()
LCC_NODES = set(Gc.nodes)

print(f'undirected graph      : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'connected components  : {len(components):,}')
print(f'largest component     : {Gc.number_of_nodes():,} nodes '
      f'({Gc.number_of_nodes() / G.number_of_nodes():.1%}), {Gc.number_of_edges():,} edges')
print(f'outside the LCC       : {G.number_of_nodes() - Gc.number_of_nodes():,} stations '
      f'(scored 0 in every lens)')

## 7. Proxy step 1: splitting each statistical area's population among its stops

The CBS provides population per **statistical area** (a neighborhood-sized polygon), not per stop. So the first step is a naive but transparent allocation: **each area's residents are split equally among the stops joined to that area**. An area with 6,000 residents and 20 stops gives each of those stops a base share of 300 residents.

This is a deliberately coarse choice, and it's coarse in a particular direction: it assumes residents are spread uniformly across the area and use its stops uniformly. It also means that **stop-dense areas dilute themselves** - two stops 80 m apart on opposite sides of the same street each get half the share, which is roughly right for a proxy but wrong in detail (to the passenger it's the same pair of stops).

Two data-quality facts from stage 09 are carried forward and reported here instead of being hidden:

* Not every stop falls inside a polygon. Stage 09 falls back to the **nearest** polygon (`socio_join_method = 'nearest'`, with the distance in `socio_join_distance_m`). A stop matched to an area 2 km away is credited with people who don't live near it at all.
* Some stops have **no CBS unit at all**, and some units **have no population figure**. Those stops get a base share of 0, meaning they're invisible as an origin/destination in the demand lens. We count them explicitly.

Where the stop-level table has no population for a unit, we fall back to the population recorded for that unit in `socioeconomic_neighborhood_access.csv`.

In [ ]:
# --- Step 1: population of a statistical area, split across its stops ------
socio = socio_df.drop_duplicates(subset='stop_id').copy()

unit_pop_from_stops = (socio.dropna(subset=['socio_unit_id'])
                            .groupby('socio_unit_id')['socio_population'].first())
unit_pop_from_access = (access_df.dropna(subset=['socio_unit_id'])
                                 .drop_duplicates(subset='socio_unit_id')
                                 .set_index('socio_unit_id')['population'])
unit_population = unit_pop_from_stops.combine_first(unit_pop_from_access)

stops_per_unit = (socio.dropna(subset=['socio_unit_id'])
                       .groupby('socio_unit_id')['stop_id'].nunique())

join_cols = ['stop_id', 'socio_unit_id', 'socio_cluster',
             'socio_join_method', 'socio_join_distance_m']
join_cols = [c for c in join_cols if c in socio.columns]

proxy = (nodes_df[['stop_id', 'stop_name', 'lat', 'lon', 'region', 'metro', 'stop_use_count']]
         .drop_duplicates(subset='stop_id')
         .merge(socio[join_cols], on='stop_id', how='left'))

proxy['unit_population'] = proxy['socio_unit_id'].map(unit_population)
proxy['unit_stops'] = proxy['socio_unit_id'].map(stops_per_unit)
proxy['area_population_share'] = (
    proxy['unit_population'] / proxy['unit_stops'].replace(0, np.nan)
).fillna(0.0)

national_population = float(unit_population.dropna().sum())
allocated = float(proxy['area_population_share'].sum())
no_unit = int(proxy['socio_unit_id'].isna().sum())
no_population = int((proxy['socio_unit_id'].notna() & proxy['unit_population'].isna()).sum())

print(f'CBS population across all areas   : {national_population:,.0f}')
print(f'population allocated to stops     : {allocated:,.0f} '
      f'({allocated / national_population:.1%} of it)')
print(f'stops with no CBS unit            : {no_unit:,}')
print(f'stops whose unit has no population: {no_population:,}')
if 'socio_join_method' in proxy.columns:
    print('\njoin method used per stop:')
    print(proxy['socio_join_method'].value_counts(dropna=False).to_string())
if 'socio_join_distance_m' in proxy.columns:
    far = int((proxy['socio_join_distance_m'] > 1000).sum())
    print(f'\nstops joined to an area more than 1 km away: {far:,} '
          f'(their population attribution is weak)')
proxy[['stop_id', 'stop_name', 'socio_unit_id', 'unit_population',
       'unit_stops', 'area_population_share']].head()

## 8. Proxy step 2: a gravity kernel over walking distance

Splitting an area's population by area membership alone creates a discontinuity at the polygon boundary: a resident 30 m from a stop is credited to it only if they happen to fall on the same side of an administrative line. The second step smooths this with a **gravity model**.

Each stop `j` redistributes its base share `p_j` across all stops within `CATCHMENT_RADIUS_M` meters of it (including itself), proportionally to `exp(-d / DECAY_LENGTH_M)`. The weights are normalized **per source**, so:

```
population_served[i] = sum_j p_j * exp(-d_ij / L) / sum_{k in radius(j)} exp(-d_jk / L)
```

Because the kernel is normalized per source, the national total is **preserved** - no resident is counted twice, which is the usual bug in radius-based catchment counts ("population within 500 m of each stop", summed over stops, can exceed the country's population several times over). What the kernel does instead is concentrate demand where stops are clustered: a lone stop serving a whole area keeps all of it, while one of twenty stops in a dense network keeps part of its own share and collects fractions from its neighbors.

Distances use an equirectangular approximation around the feed's mean latitude, accurate to well under a meter at these scales, and stops are binned into a grid of cells sized `CATCHMENT_RADIUS_M`, so only the 3x3 neighborhood of cells is scanned. The cost: a few seconds to a minute for ~30k stops. Stops with missing coordinates keep their base share and take part in no exchange.

In [ ]:
# --- Step 2: distance-decayed re-allocation (population-conserving) --------
def allocate_population(lat, lon, base_pop, radius_m, decay_m):
    """Spread each stop's base population over stops within `radius_m`.

    Weights are exp(-d / decay_m), normalised per source, so the sum of the
    returned array equals the sum of `base_pop` (no double counting).
    """
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    base_pop = np.asarray(base_pop, dtype=float)
    n = len(lat)
    served = np.zeros(n, dtype=float)

    valid = np.isfinite(lat) & np.isfinite(lon)
    m_per_deg_lat = 110574.0
    m_per_deg_lon = 111320.0 * math.cos(math.radians(float(np.nanmean(lat[valid]))))

    x = lon * m_per_deg_lon
    y = lat * m_per_deg_lat

    # bucket stops into square cells of side `radius_m`
    cells = {}
    ci = np.zeros(n, dtype=np.int64)
    cj = np.zeros(n, dtype=np.int64)
    for idx in np.flatnonzero(valid):
        ci[idx] = int(math.floor(y[idx] / radius_m))
        cj[idx] = int(math.floor(x[idx] / radius_m))
        cells.setdefault((ci[idx], cj[idx]), []).append(idx)
    cells = {key: np.asarray(val, dtype=np.int64) for key, val in cells.items()}

    neighbour_cache = {}

    def candidates(key):
        got = neighbour_cache.get(key)
        if got is None:
            parts = [cells[(key[0] + a, key[1] + b)]
                     for a in (-1, 0, 1) for b in (-1, 0, 1)
                     if (key[0] + a, key[1] + b) in cells]
            got = np.concatenate(parts) if parts else np.empty(0, dtype=np.int64)
            neighbour_cache[key] = got
        return got

    for idx in np.flatnonzero(base_pop > 0):
        if not valid[idx]:
            served[idx] += base_pop[idx]      # no coordinates: keep it in place
            continue
        cand = candidates((ci[idx], cj[idx]))
        d = np.hypot(x[cand] - x[idx], y[cand] - y[idx])
        keep = d <= radius_m
        cand, d = cand[keep], d[keep]
        if cand.size == 0:
            served[idx] += base_pop[idx]
            continue
        w = np.exp(-d / decay_m)
        total = w.sum()
        if not np.isfinite(total) or total <= 0:
            served[idx] += base_pop[idx]
            continue
        np.add.at(served, cand, base_pop[idx] * (w / total))
    return served


t0 = time.time()
proxy['population_served'] = allocate_population(
    proxy['lat'].to_numpy(), proxy['lon'].to_numpy(),
    proxy['area_population_share'].to_numpy(),
    CATCHMENT_RADIUS_M, DECAY_LENGTH_M,
)
print(f'gravity allocation (radius={CATCHMENT_RADIUS_M:.0f} m, '
      f'decay={DECAY_LENGTH_M:.0f} m) in {time.time() - t0:.1f}s')
print(f'conservation check : allocated {proxy["area_population_share"].sum():,.0f} -> '
      f'served {proxy["population_served"].sum():,.0f}')
print(f'stops with zero population PROXY: '
      f'{int((proxy["population_served"] <= 0).sum()):,} of {len(proxy):,}')
print(proxy['population_served'].describe().round(1).to_string())

## 9. Proxy step 3: folding in service volume, and the contract table

Population alone tells you how many people live near a stop, not how many of them can plausibly *use* it. A stop served by four trips a day and one served by four hundred don't attract the same share of their neighborhood's trips. So we multiply by a service term:

```
demand_weight_raw[i] = population_served[i] * (stop_use_count[i] / median_stop_use_count) ** SERVICE_EXPONENT
demand_weight[i]     = demand_weight_raw[i] / sum(demand_weight_raw)      # sums to 1
```

with `SERVICE_EXPONENT = 0.5` by default - a square-root-style blend, chosen so service volume tempers the proxy without dominating it.

**This step is a compromise, and it cuts both ways.** Including service volume makes the proxy more realistic (people do use stops that have buses) but partially imports back the very topology we're trying to weight *aside*: `stop_use_count` is strongly correlated with weighted degree. So section 14 repeats the whole analysis with `SERVICE_EXPONENT = 0` (pure population, fully independent of service), so the reader can see exactly how much of the result comes from the population signal and how much from the service signal.

This cell writes the first contract file, `tables/demand_proxy.csv`, with exactly the three agreed columns `stop_id, population_served, demand_weight`. Every intermediate column, plus the CBS join quality per stop, is written to `tables/demand_proxy_detail.csv` so nothing is lost.

In [ ]:
# --- Step 3: demand weight = population PROXY x service volume -------------
def build_demand_weight(population_served, stop_use_count, service_exponent):
    """Return weights summing to 1 (or all-zero if no population is available)."""
    pop = np.asarray(population_served, dtype=float)
    use = pd.to_numeric(pd.Series(stop_use_count), errors='coerce').fillna(0.0).to_numpy()
    positive = use[use > 0]
    reference = float(np.median(positive)) if positive.size else 1.0
    relative = np.where(use > 0, use / reference, 1e-6)
    raw = pop * np.power(relative, service_exponent)
    raw = np.where(np.isfinite(raw) & (raw > 0), raw, 0.0)
    total = raw.sum()
    return raw / total if total > 0 else raw


proxy['demand_weight'] = build_demand_weight(
    proxy['population_served'], proxy['stop_use_count'], SERVICE_EXPONENT)

# --- contract table: exactly the three agreed columns ----------------------
(proxy[['stop_id', 'population_served', 'demand_weight']]
    .sort_values('demand_weight', ascending=False)
    .to_csv(TABLES / 'demand_proxy.csv', index=False, encoding='utf-8-sig'))

detail_cols = [c for c in ['stop_id', 'stop_name', 'lat', 'lon', 'region', 'metro',
                           'socio_unit_id', 'socio_cluster', 'socio_join_method',
                           'socio_join_distance_m', 'unit_population', 'unit_stops',
                           'area_population_share', 'stop_use_count',
                           'population_served', 'demand_weight'] if c in proxy.columns]
(proxy[detail_cols]
    .sort_values('demand_weight', ascending=False)
    .to_csv(TABLES / 'demand_proxy_detail.csv', index=False, encoding='utf-8-sig'))

share_top1pct = float(proxy['demand_weight'].nlargest(max(1, len(proxy) // 100)).sum())
rho_pop_service = float(pd.Series(proxy['population_served']).corr(
    pd.to_numeric(proxy['stop_use_count'], errors='coerce'), method='spearman'))

print(f'demand_proxy.csv written ({len(proxy):,} rows) -> {TABLES / "demand_proxy.csv"}')
print(f'top 1% of stops hold {share_top1pct:.1%} of the total PROXY demand weight')
print(f'Spearman(population PROXY, service volume) = {rho_pop_service:.3f}')
print(f'stops with zero PROXY demand weight: '
      f'{int((proxy["demand_weight"] <= 0).sum()):,} (invisible as origins/destinations)')
proxy.nlargest(TOP_N, 'demand_weight')[
    ['stop_name', 'region', 'socio_cluster', 'stop_use_count',
     'population_served', 'demand_weight']].round(5)

## 10. What the proxy looks like

Three views, all marked as proxy:

1. **The distribution** of `population_served`. A long right tail is expected - a handful of stops sit alone in a dense area and inherit thousands of residents, while most share their area with a dozen neighbors.
2. **Service volume against the population proxy**. This scatter is the sanity check for section 9: if the two were nearly perfectly correlated, the demand lens would be a service ranking under another name and the whole exercise would be circular. The Spearman rho printed above tells you how much independent signal the population term actually adds.
3. **A map** of the proxy, on a logarithmic color scale. It should look like a population map of Israel - dense along the coastal plain and in Jerusalem, sparse in the Negev - and *not* like a map of the bus network. If it looked like the latter, the proxy would be measuring service, not people.

In [ ]:
# --- Figures: what the demand PROXY looks like -----------------------------
served = proxy['population_served'].astype(float)
use = pd.to_numeric(proxy['stop_use_count'], errors='coerce').fillna(0.0)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
axes[0].hist(served[served > 0], bins=60, color='#0891b2')
axes[0].set_yscale('log')
axes[0].set_xlabel('PROXY population served (residents allocated per stop)')
axes[0].set_ylabel('Stations (log scale)')
axes[0].set_title('Population PROXY per stop (NOT ridership)')

axes[1].scatter(use.where(use > 0), served, s=4, alpha=0.25, color='#7c3aed')
axes[1].set_xscale('log')
axes[1].set_xlabel('Scheduled stop visits per stop (service volume)')
axes[1].set_ylabel('PROXY population served')
axes[1].set_title(f'Service volume vs population PROXY (Spearman rho = {rho_pop_service:.2f})')
fig.tight_layout()
fig.savefig(FIGURES / 'demand_proxy_distribution.png', dpi=FIG_DPI)
plt.show()

geo = proxy.dropna(subset=['lat', 'lon']).copy()
geo = geo[geo['demand_weight'] > 0]
if len(geo):
    fig, ax = plt.subplots(figsize=(7.5, 10))
    sc = ax.scatter(geo['lon'], geo['lat'], s=5,
                    c=np.log10(geo['demand_weight'] + 1e-12),
                    cmap='viridis', alpha=0.7)
    fig.colorbar(sc, ax=ax, shrink=0.7, label='log10 PROXY demand weight')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('PROXY demand weight per stop\n(population-based estimate, not measured ridership)')
    fig.tight_layout()
    fig.savefig(FIGURES / 'demand_proxy_map.png', dpi=FIG_DPI)
    plt.show()
else:
    print('No stop has both coordinates and a positive PROXY weight - skipping the map.')

## 11. The demand-weighted betweenness estimator

Standard betweenness treats every ordered pair of stops as one unit of traffic. We want

```
BC_demand(v) = sum over s != v != t of  w(s) * w(t) * sigma_st(v) / sigma_st
```

where `w` is the PROXY demand weight: a shortest path counts for the number of people plausibly at its two ends. `networkx` can't do this (its `betweenness_centrality` has no endpoint weighting), so the cell below implements the Brandes algorithm directly, with two changes:

* **Destination weighting.** In the backward accumulation, the usual `(1 + delta[w])` becomes `(w(t) + delta[w])`, making each destination contribute proportionally to its demand weight. This is the canonical Brandes generalization for weighted endpoints and costs nothing.
* **Demand-proportional source sampling.** Summing over all ~30k sources is hours of work. Instead we draw `K_DEMAND_SOURCES` sources **with replacement, with probability `w(s) / sum(w)`**, accumulate the dependencies, and rescale by `sum(w) / k`. Because the sampling probability is exactly the source weight, this is an unbiased estimator of the sum above - the source weighting comes "for free" from the sampling distribution, with no importance-weighting correction needed.

Setting `w = 1` everywhere reduces the estimator to ordinary sampled betweenness (up to a constant `n`, which doesn't affect the rankings). That gives us the **uniform control** lens: same code, same seed regime, same source count, so any difference between the two lenses is caused by the demand weighting and nothing else.

**Cost.** One BFS plus one accumulation over the largest component per sampled source: about 0.2-0.5 s each in pure Python, so about 1.5-4 minutes per lens at `k = 400`. The estimate is unbiased at any `k`, but individual scores - especially in the tail - are noisy, exactly as notebook 04's `approx_betweenness` is noisy. Rankings near the top are far more stable than rankings in the middle.

In [ ]:
# --- Brandes betweenness with demand-weighted endpoints, sampled sources ---
def sampled_endpoint_weighted_betweenness(graph, node_weight, k, seed,
                                          progress_every=None, label=''):
    """Estimate sum_{s,t} w(s) w(t) sigma_st(v) / sigma_st for every v.

    Sources are drawn with replacement with probability proportional to
    `node_weight`; targets contribute their own weight inside the Brandes
    accumulation. The estimate is unbiased for any k.
    """
    nodes = list(graph.nodes)
    weights = np.array([float(node_weight.get(n, 0.0)) for n in nodes], dtype=float)
    weights[~np.isfinite(weights)] = 0.0
    weights[weights < 0] = 0.0
    total_weight = float(weights.sum())
    if total_weight <= 0:
        raise ValueError('every node weight is zero - cannot sample sources')

    rng = np.random.default_rng(seed)
    draws = rng.choice(len(nodes), size=int(k), replace=True, p=weights / total_weight)
    target_weight = {n: float(weights[i]) for i, n in enumerate(nodes)}
    scores = dict.fromkeys(nodes, 0.0)

    t_start = time.time()
    for step, source_index in enumerate(draws, start=1):
        s = nodes[source_index]
        # ---- forward pass: BFS shortest-path DAG (hop distance) ----
        stack = []
        preds = {s: []}
        sigma = {s: 1.0}
        dist = {s: 0}
        queue = deque([s])
        while queue:
            v = queue.popleft()
            stack.append(v)
            dist_v = dist[v]
            sigma_v = sigma[v]
            for nbr in graph[v]:
                if nbr not in dist:
                    dist[nbr] = dist_v + 1
                    sigma[nbr] = 0.0
                    preds[nbr] = []
                    queue.append(nbr)
                if dist[nbr] == dist_v + 1:
                    sigma[nbr] += sigma_v
                    preds[nbr].append(v)
        # ---- backward pass: dependency accumulation, targets weighted ----
        delta = dict.fromkeys(stack, 0.0)
        while stack:
            w_node = stack.pop()
            coeff = (target_weight[w_node] + delta[w_node]) / sigma[w_node]
            for v in preds[w_node]:
                delta[v] += sigma[v] * coeff
            if w_node != s:
                scores[w_node] += delta[w_node]
        if progress_every and step % progress_every == 0:
            elapsed = time.time() - t_start
            print(f'  [{label}] {step}/{len(draws)} sources, {elapsed:.0f}s elapsed, '
                  f'~{elapsed / step * (len(draws) - step):.0f}s left')

    scale = total_weight / len(draws)
    meta = {'k': int(len(draws)), 'seed': int(seed),
            'total_weight': total_weight,
            'distinct_sources': int(len(set(draws.tolist()))),
            'seconds': round(time.time() - t_start, 1)}
    return {n: value * scale for n, value in scores.items()}, meta


print('estimator defined')

## 12. Running the two lenses

We now run the estimator twice on the largest connected component:

* **Uniform control** - weight 1 on every node. This is ordinary sampled betweenness, and it exists so the comparison against the demand lens is a fair apples-to-apples one. Comparing the demand lens directly against notebook 04's column would mix two differences at once (different weighting *and* a different source sample), so we report both comparisons and treat the control as the primary one.
* **Demand-weighted (PROXY)** - the node weights are `demand_weight`. Stops with weight 0 (no CBS population, or outside the CBS join) are never sampled as a source and contribute nothing as a destination, though they can still serve as an intermediary and receive a score. This is a real limitation, and section 15 says so.

This is the expensive cell: about 3-8 minutes total at `K_DEMAND_SOURCES = 400`. Progress lines with an estimated time remaining are printed every `PROGRESS_EVERY` sources.

In [ ]:
# --- Run the uniform control lens and the demand-weighted PROXY lens -------
demand_lookup = dict(zip(proxy['stop_id'], proxy['demand_weight'].astype(float)))
demand_weight_lcc = {n: demand_lookup.get(n, 0.0) for n in Gc.nodes}
uniform_weight_lcc = {n: 1.0 for n in Gc.nodes}

covered = sum(1 for v in demand_weight_lcc.values() if v > 0)
print(f'LCC stations with a positive PROXY weight: {covered:,} of {len(demand_weight_lcc):,} '
      f'({covered / len(demand_weight_lcc):.1%})')
print(f'share of the national PROXY weight inside the LCC: '
      f'{sum(demand_weight_lcc.values()):.3%}\n')

print('running the uniform-weight control lens ...')
uniform_scores, uniform_meta = sampled_endpoint_weighted_betweenness(
    Gc, uniform_weight_lcc, K_DEMAND_SOURCES, SEED_UNIFORM,
    progress_every=PROGRESS_EVERY, label='uniform')
print(f'  done in {uniform_meta["seconds"]}s '
      f'({uniform_meta["distinct_sources"]} distinct sources)\n')

print('running the demand-weighted PROXY lens ...')
demand_scores, demand_meta = sampled_endpoint_weighted_betweenness(
    Gc, demand_weight_lcc, K_DEMAND_SOURCES, SEED_DEMAND,
    progress_every=PROGRESS_EVERY, label='demand')
print(f'  done in {demand_meta["seconds"]}s '
      f'({demand_meta["distinct_sources"]} distinct sources)')

## 13. Assembling the rankings and writing the contract table

Three score columns are attached to one table, one per lens:

* `topological_betweenness_nb04` - notebook 04's `approx_betweenness` column, the project's existing pure topological ranking;
* `uniform_control_betweenness` - the control from the same estimator;
* `demand_weighted_betweenness_proxy` - the demand lens (the column name carries the word `proxy` on purpose).

Each is converted to a rank where **1 = the most critical**, using average ranks for ties (many stops get a score of exactly 0 in a sampled estimate, and averaging keeps that tie block from creating a fake order).

`rank_shift = topological_rank - demand_weighted_rank`, so a **positive shift means the stop is more important once people are taken into account**, and a negative shift means the topological ranking overrated it. The contract file `tables/demand_weighted_criticality.csv` carries exactly `stop_id, stop_name, topological_rank, demand_weighted_rank, rank_shift`; everything else (scores, coordinates, socioeconomic cluster, the control lens and its rank) is written to `tables/criticality_lens_detail.csv`.

In [ ]:
# --- Join the three lenses, rank them, write the contract table ------------
lenses = proxy[['stop_id', 'stop_name', 'lat', 'lon', 'region', 'metro',
                'socio_cluster', 'stop_use_count', 'population_served',
                'demand_weight']].copy()

nb04_betweenness = dict(zip(metrics_df['stop_id'],
                            pd.to_numeric(metrics_df['approx_betweenness'],
                                          errors='coerce').fillna(0.0)))
lenses['topological_betweenness_nb04'] = lenses['stop_id'].map(nb04_betweenness).fillna(0.0)
lenses['uniform_control_betweenness'] = lenses['stop_id'].map(uniform_scores).fillna(0.0)
lenses['demand_weighted_betweenness_proxy'] = lenses['stop_id'].map(demand_scores).fillna(0.0)
lenses['in_largest_component'] = lenses['stop_id'].isin(LCC_NODES)

RANK_OF = {
    'topological_betweenness_nb04': 'topological_rank',
    'uniform_control_betweenness': 'uniform_control_rank',
    'demand_weighted_betweenness_proxy': 'demand_weighted_rank',
}
for score_col, rank_col in RANK_OF.items():
    lenses[rank_col] = lenses[score_col].rank(ascending=False, method='average')

lenses['rank_shift'] = lenses['topological_rank'] - lenses['demand_weighted_rank']
lenses['rank_shift_vs_control'] = lenses['uniform_control_rank'] - lenses['demand_weighted_rank']
lenses['label'] = lenses['stop_name'].fillna('').astype(str).str.strip()
lenses['label'] = lenses['label'].where(lenses['label'] != '', lenses['stop_id'])

# --- contract table: exactly the five agreed columns -----------------------
(lenses[['stop_id', 'stop_name', 'topological_rank', 'demand_weighted_rank', 'rank_shift']]
    .sort_values('demand_weighted_rank')
    .to_csv(TABLES / 'demand_weighted_criticality.csv', index=False, encoding='utf-8-sig'))

(lenses.drop(columns=['label'])
       .sort_values('demand_weighted_rank')
       .to_csv(TABLES / 'criticality_lens_detail.csv', index=False, encoding='utf-8-sig'))

print(f'demand_weighted_criticality.csv written ({len(lenses):,} rows)')
print(f'stations with a non-zero demand-weighted score: '
      f'{int((lenses["demand_weighted_betweenness_proxy"] > 0).sum()):,}')
print(f'stations with a non-zero uniform control score: '
      f'{int((lenses["uniform_control_betweenness"] > 0).sum()):,}\n')
print('Top stations under the demand-weighted PROXY lens:')
lenses.nsmallest(TOP_N, 'demand_weighted_rank')[
    ['stop_name', 'region', 'socio_cluster', 'population_served',
     'topological_rank', 'demand_weighted_rank', 'rank_shift']].round(1)

## 14. Do the two rankings agree?

Three summary statistics per lens pair:

* **Spearman rho over all stops.** Expect this to be high, and don't read too much into it: about 90% of stops have a small but nonzero betweenness, and their relative order barely moves, which inflates the correlation. This is a floor, not a finding.
* **Spearman rho over the union of the two top-1000 sets.** Restricting to stops that at least one lens considers important is the more informative number, because that's the population a robustness study would actually act on.
* **Top-N overlap** for N in `OVERLAP_SIZES`. "How many of the 50 stops you'd protect by topological ranking are still among the 50 stops you'd protect after demand weighting." This is the number the report should cite.

The primary comparison is **uniform control against the demand PROXY**, because both come from the same estimator with the same `k`, so the difference is the weighting alone. The comparison against notebook 04 is also reported, but it additionally absorbs the sampling noise of two independent 300/400-source estimates, so any disagreement there is an upper bound on the true effect of demand weighting.

In [ ]:
# --- Agreement between the lenses ------------------------------------------
PAIRS = [
    ('uniform_control_betweenness', 'demand_weighted_betweenness_proxy',
     'uniform control (same estimator)', 'demand-weighted PROXY'),
    ('topological_betweenness_nb04', 'demand_weighted_betweenness_proxy',
     'topological nb04', 'demand-weighted PROXY'),
    ('topological_betweenness_nb04', 'uniform_control_betweenness',
     'topological nb04', 'uniform control (same estimator)'),
]

rows = []
for col_a, col_b, name_a, name_b in PAIRS:
    a, b = lenses[col_a], lenses[col_b]
    top_union = set(lenses.nlargest(1000, col_a).index) | set(lenses.nlargest(1000, col_b).index)
    sub = lenses.loc[sorted(top_union)]
    row = {
        'lens_a': name_a,
        'lens_b': name_b,
        'spearman_all_stations': round(float(a.corr(b, method='spearman')), 4),
        'spearman_top1000_union': round(float(sub[col_a].corr(sub[col_b], method='spearman')), 4),
    }
    for n in OVERLAP_SIZES:
        set_a = set(lenses.nlargest(n, col_a)['stop_id'])
        set_b = set(lenses.nlargest(n, col_b)['stop_id'])
        row[f'top{n}_overlap'] = round(len(set_a & set_b) / n, 4)
    rows.append(row)

agreement = pd.DataFrame(rows)
agreement.to_csv(TABLES / 'rank_agreement.csv', index=False, encoding='utf-8-sig')
display(agreement)

ranked = lenses[lenses['in_largest_component']].copy()
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))
axes[0].scatter(ranked['uniform_control_rank'], ranked['demand_weighted_rank'],
                s=4, alpha=0.2, color='#2563eb')
lim = float(max(ranked['uniform_control_rank'].max(), ranked['demand_weighted_rank'].max()))
axes[0].plot([1, lim], [1, lim], ls='--', lw=1, color='#334155', label='no change')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].invert_xaxis()
axes[0].invert_yaxis()
axes[0].set_xlabel('Rank under the uniform control lens (1 = most critical)')
axes[0].set_ylabel('Rank under the demand-weighted PROXY lens')
axes[0].set_title('Same estimator, different endpoint weights')
axes[0].legend(loc='lower right')

axes[1].scatter(ranked['topological_rank'], ranked['demand_weighted_rank'],
                s=4, alpha=0.2, color='#dc2626')
lim2 = float(max(ranked['topological_rank'].max(), ranked['demand_weighted_rank'].max()))
axes[1].plot([1, lim2], [1, lim2], ls='--', lw=1, color='#334155', label='no change')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].invert_xaxis()
axes[1].invert_yaxis()
axes[1].set_xlabel('Topological rank (notebook 04 approx_betweenness)')
axes[1].set_ylabel('Rank under the demand-weighted PROXY lens')
axes[1].set_title('Topology vs demand PROXY (also absorbs sampling noise)')
axes[1].legend(loc='lower right')
fig.tight_layout()
fig.savefig(FIGURES / 'rank_rank_scatter.png', dpi=FIG_DPI)
plt.show()

## 15. Which stations move - the actual result

The interesting output of this notebook isn't a correlation coefficient, it's a **list of names**. This section extracts the stations whose rank moves the most between the topological lens and the demand PROXY lens.

Two guards keep the list from being an artifact:

1. Only stations **inside the largest connected component** and with a nonzero score in at least one lens qualify - otherwise the huge tie block of zero-score stations would produce meaningless "movement".
2. Only stations reaching the **top 2,000 in at least one lens** are considered. A station moving from rank 24,000 to rank 19,000 is noise; a station moving from rank 1,900 to rank 120 is the phenomenon we're after.

`rank_movers.csv` records both directions: **gainers** (positive `rank_shift`: stations the topological view underrates, usually low-degree stops embedded in dense residential areas) and **losers** (negative `rank_shift`: well-connected nodes in sparsely populated places - interchange nodes on intercity corridors, stops in industrial zones, nodes in the desert). Remember that both lenses are sampled estimates, so a shift of a few hundred places for a single station is within the noise; the pattern across the whole list is what carries meaning.

In [ ]:
# --- Biggest gainers and losers of rank ------------------------------------
eligible = lenses[
    lenses['in_largest_component']
    & ((lenses['topological_betweenness_nb04'] > 0)
       | (lenses['demand_weighted_betweenness_proxy'] > 0))
    & ((lenses['topological_rank'] <= 2000) | (lenses['demand_weighted_rank'] <= 2000))
].copy()

gainers = eligible.nlargest(MOVERS_N, 'rank_shift').copy()
losers = eligible.nsmallest(MOVERS_N, 'rank_shift').copy()
gainers['direction'] = 'gains under demand PROXY'
losers['direction'] = 'loses under demand PROXY'

MOVER_COLS = ['direction', 'stop_id', 'stop_name', 'region', 'metro', 'socio_cluster',
              'population_served', 'stop_use_count', 'topological_rank',
              'demand_weighted_rank', 'uniform_control_rank', 'rank_shift',
              'rank_shift_vs_control']
movers = pd.concat([gainers, losers])[MOVER_COLS]
movers.to_csv(TABLES / 'rank_movers.csv', index=False, encoding='utf-8-sig')

print(f'eligible stations for the movers analysis: {len(eligible):,}')
print(f'median |rank shift| among them            : '
      f'{eligible["rank_shift"].abs().median():,.0f} places\n')
print('Biggest GAINERS (topology under-rates them once population is weighted in):')
display(gainers[['stop_name', 'region', 'socio_cluster', 'population_served',
                 'stop_use_count', 'topological_rank', 'demand_weighted_rank',
                 'rank_shift']].round(1).head(TOP_N))
print('Biggest LOSERS (topologically central, but few residents nearby):')
display(losers[['stop_name', 'region', 'socio_cluster', 'population_served',
                'stop_use_count', 'topological_rank', 'demand_weighted_rank',
                'rank_shift']].round(1).head(TOP_N))

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, frame, colour, title in [
    (axes[0], gainers.head(TOP_N), '#16a34a',
     f'Top {TOP_N} rank GAINERS under the demand PROXY'),
    (axes[1], losers.head(TOP_N), '#dc2626',
     f'Top {TOP_N} rank LOSERS under the demand PROXY'),
]:
    frame = frame.sort_values('rank_shift')
    y = np.arange(len(frame))
    ax.barh(y, frame['rank_shift'].astype(float).values, color=colour)
    ax.set_yticks(y)
    ax.set_yticklabels(frame['stop_name'].fillna('').astype(str)
                            .where(frame['stop_name'].notna(), frame['stop_id']).values)
    ax.set_xlabel('Rank shift (topological rank - demand PROXY rank)')
    ax.set_title(title)
fig.tight_layout()
fig.savefig(FIGURES / 'rank_movers.png', dpi=FIG_DPI)
plt.show()

## 16. Does demand weighting change *who* the network protects?

This is the equity question notebook 09 raised, asked from the robustness side. If demand weighting systematically promotes stations in low socioeconomic clusters, then a pure topological criticality ranking - the kind an operator would use to prioritize investment in redundancy - is quietly biased against poorer neighborhoods, because those neighborhoods are dense in people but not necessarily dense in network structure. If the mean rank shift is flat across the clusters, this specific concern isn't supported by the data.

We report the mean and median `rank_shift` per CBS cluster (1 = weakest, 10 = strongest), the number of stations per cluster, and how each cluster's share of the top 200 changes between the two lenses. Note the confounder before reading the result: cluster and population **density** are correlated in Israel, and the proxy is built from density, so part of any pattern here is mechanical rather than a discovery about public-transport policy.

In [ ]:
# --- Rank movement by CBS socioeconomic cluster ----------------------------
cluster_frame = lenses.dropna(subset=['socio_cluster']).copy()
cluster_frame['socio_cluster'] = cluster_frame['socio_cluster'].astype(float).round().astype(int)

top_topo = set(lenses.nsmallest(200, 'topological_rank')['stop_id'])
top_demand = set(lenses.nsmallest(200, 'demand_weighted_rank')['stop_id'])
cluster_frame['in_top200_topological'] = cluster_frame['stop_id'].isin(top_topo)
cluster_frame['in_top200_demand'] = cluster_frame['stop_id'].isin(top_demand)

by_cluster = (cluster_frame
              .groupby('socio_cluster')
              .agg(stations=('stop_id', 'size'),
                   mean_rank_shift=('rank_shift', 'mean'),
                   median_rank_shift=('rank_shift', 'median'),
                   mean_population_served=('population_served', 'mean'),
                   top200_topological=('in_top200_topological', 'sum'),
                   top200_demand=('in_top200_demand', 'sum'))
              .reset_index())
by_cluster['top200_change'] = by_cluster['top200_demand'] - by_cluster['top200_topological']
by_cluster = by_cluster.round(2)
by_cluster.to_csv(TABLES / 'rank_shift_by_socioeconomic_cluster.csv',
                  index=False, encoding='utf-8-sig')
display(by_cluster)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours = ['#16a34a' if v >= 0 else '#dc2626' for v in by_cluster['mean_rank_shift']]
axes[0].bar(by_cluster['socio_cluster'].astype(str), by_cluster['mean_rank_shift'], color=colours)
axes[0].axhline(0, color='#334155', lw=1)
axes[0].set_xlabel('CBS socioeconomic cluster (1 = weakest, 10 = strongest)')
axes[0].set_ylabel('Mean rank shift (+ = promoted)')
axes[0].set_title('Who gains when criticality is weighted by the demand PROXY')

width = 0.4
pos = np.arange(len(by_cluster))
axes[1].bar(pos - width / 2, by_cluster['top200_topological'], width,
            label='top 200, topological', color='#64748b')
axes[1].bar(pos + width / 2, by_cluster['top200_demand'], width,
            label='top 200, demand PROXY', color='#0891b2')
axes[1].set_xticks(pos)
axes[1].set_xticklabels(by_cluster['socio_cluster'].astype(str))
axes[1].set_xlabel('CBS socioeconomic cluster')
axes[1].set_ylabel('Stations in the top 200')
axes[1].set_title('Composition of the top 200 under each lens')
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES / 'rank_shift_by_socioeconomic_cluster.png', dpi=FIG_DPI)
plt.show()

## 17. How much do the proxy's arbitrary parameters matter?

Three numbers in this notebook were chosen by judgment rather than measured: the catchment radius, the decay length, and the service exponent. A result that survives only under one setting isn't a result. Two sensitivity blocks:

* **At the weight level (cheap, `RUN_WEIGHT_SENSITIVITY`).** Rebuild `demand_weight` for every combination of `RADIUS_GRID` x `SERVICE_EXPONENT_GRID` (the decay length varies with the radius) and report the Spearman correlation of each variant against the base weights. A few seconds per radius.
* **At the betweenness level (expensive, `RUN_BETWEENNESS_SENSITIVITY`).** Rerun the estimator itself with `SENSITIVITY_K = 150` sources for two alternative proxies - **pure population** (`SERVICE_EXPONENT = 0`, no service term at all, so the weighting is fully independent of the network) and **wide catchment** (radius 1 km, decay 500 m) - and report how the ranking compares to the base demand lens. This costs about 1-3 more minutes; the pure-population variant is the important one, because it shows how much of the demand ranking is actually driven by service volume creeping back in.

Note that a sensitivity run at `k = 150` is noisier than the `k = 400` baseline, so part of any disagreement it shows is estimator noise rather than a real parameter effect. The gap between `k = 150` and `k = 400` sets the floor: two runs of the *same* model at these sample sizes would already differ from each other somewhat.

In [ ]:
# --- Sensitivity of the PROXY to its arbitrary parameters ------------------
baseline_weight = proxy['demand_weight'].to_numpy()
weight_rows = []

if RUN_WEIGHT_SENSITIVITY:
    served_by_radius = {}
    for radius in RADIUS_GRID:
        decay = radius / 2.0
        t0 = time.time()
        served_by_radius[radius] = allocate_population(
            proxy['lat'].to_numpy(), proxy['lon'].to_numpy(),
            proxy['area_population_share'].to_numpy(), radius, decay)
        print(f'  radius {radius:.0f} m / decay {decay:.0f} m: {time.time() - t0:.1f}s')
    for radius in RADIUS_GRID:
        for exponent in SERVICE_EXPONENT_GRID:
            variant = build_demand_weight(served_by_radius[radius],
                                          proxy['stop_use_count'], exponent)
            weight_rows.append({
                'catchment_radius_m': radius,
                'decay_length_m': radius / 2.0,
                'service_exponent': exponent,
                'is_baseline': bool(radius == CATCHMENT_RADIUS_M
                                    and exponent == SERVICE_EXPONENT),
                'spearman_vs_baseline_weight': round(float(
                    pd.Series(variant).corr(pd.Series(baseline_weight), method='spearman')), 4),
                'top200_overlap_vs_baseline': round(float(len(
                    set(pd.Series(variant, index=proxy['stop_id']).nlargest(200).index)
                    & set(pd.Series(baseline_weight, index=proxy['stop_id']).nlargest(200).index)
                ) / 200), 4),
            })
    weight_sensitivity = pd.DataFrame(weight_rows)
    weight_sensitivity.to_csv(TABLES / 'proxy_sensitivity.csv',
                              index=False, encoding='utf-8-sig')
    display(weight_sensitivity)
else:
    weight_sensitivity = pd.DataFrame()
    print('RUN_WEIGHT_SENSITIVITY is False - skipped.')

### 17b. Sensitivity of the ranking itself

The cell above shows only that the *weights* move. What matters is whether the *ranking* moves. This cell reruns the estimator on the two alternative proxies and compares the resulting rankings to the base demand lens, using the same Spearman / top-N overlap statistics as section 14. It also reruns the **base** proxy at `SENSITIVITY_K` sources, which gives the noise floor: how much disagreement comes from changing the sample size and seed alone.

In [ ]:
# --- Does the RANKING move when the PROXY definition moves? ----------------
betweenness_sensitivity = pd.DataFrame()
if RUN_BETWEENNESS_SENSITIVITY:
    baseline_series = lenses.set_index('stop_id')['demand_weighted_betweenness_proxy']

    served_pure = proxy['population_served'].to_numpy()
    served_wide = allocate_population(proxy['lat'].to_numpy(), proxy['lon'].to_numpy(),
                                      proxy['area_population_share'].to_numpy(), 1000.0, 500.0)
    VARIANTS = [
        ('noise_floor_baseline_k150', build_demand_weight(
            served_pure, proxy['stop_use_count'], SERVICE_EXPONENT), 101),
        ('pure_population_no_service', build_demand_weight(
            served_pure, proxy['stop_use_count'], 0.0), 102),
        ('wide_catchment_1km', build_demand_weight(
            served_wide, proxy['stop_use_count'], SERVICE_EXPONENT), 103),
    ]

    rows = []
    for name, weight_vector, seed in VARIANTS:
        lookup = dict(zip(proxy['stop_id'], weight_vector))
        weights_lcc = {n: float(lookup.get(n, 0.0)) for n in Gc.nodes}
        print(f'running variant "{name}" at k={SENSITIVITY_K} ...')
        scores, meta = sampled_endpoint_weighted_betweenness(
            Gc, weights_lcc, SENSITIVITY_K, seed,
            progress_every=PROGRESS_EVERY, label=name)
        variant_series = pd.Series(scores).reindex(baseline_series.index).fillna(0.0)
        row = {
            'variant': name,
            'k_sources': meta['k'],
            'seconds': meta['seconds'],
            'spearman_vs_baseline_lens': round(float(
                variant_series.corr(baseline_series, method='spearman')), 4),
        }
        for n in OVERLAP_SIZES:
            row[f'top{n}_overlap_vs_baseline'] = round(float(len(
                set(variant_series.nlargest(n).index) & set(baseline_series.nlargest(n).index)
            ) / n), 4)
        rows.append(row)
        print(f'  {name}: rho = {row["spearman_vs_baseline_lens"]}, '
              f'top50 overlap = {row["top50_overlap_vs_baseline"]}\n')

    betweenness_sensitivity = pd.DataFrame(rows)
    betweenness_sensitivity.to_csv(TABLES / 'betweenness_sensitivity.csv',
                                   index=False, encoding='utf-8-sig')
    display(betweenness_sensitivity)
else:
    print('RUN_BETWEENNESS_SENSITIVITY is False - skipped. '
          'The parameter robustness of the ranking is then untested.')

## 18. Summary JSON

Everything a later notebook or the report might want to cite, in one dictionary: the proxy parameters, its coverage and its data-quality caveats, the estimator settings, the agreement statistics between the lenses, and the biggest gainer and loser. The flag `proxy_is_not_ridership` is written as `true` on purpose, so any consumer of this file who forgets the caveat gets a reminder from the data itself.

In [ ]:
# --- demand_summary.json ---------------------------------------------------
def _agreement(name_a, name_b, key):
    hit = agreement[(agreement['lens_a'] == name_a) & (agreement['lens_b'] == name_b)]
    return float(hit.iloc[0][key]) if len(hit) else None


top_gainer = gainers.iloc[0] if len(gainers) else None
top_loser = losers.iloc[0] if len(losers) else None

summary = {
    'stage': '21_demand_weighted_criticality',
    'proxy_is_not_ridership': True,
    'proxy_description': ('CBS 2021 statistical-area population split across the stops of '
                          'each area, re-allocated with a distance-decayed gravity kernel, '
                          'then multiplied by scheduled service volume. GTFS contains no '
                          'boarding, alighting or ridership data of any kind.'),
    'parameters': {
        'catchment_radius_m': CATCHMENT_RADIUS_M,
        'decay_length_m': DECAY_LENGTH_M,
        'service_exponent': SERVICE_EXPONENT,
        'k_sources_per_lens': K_DEMAND_SOURCES,
        'seed_demand': SEED_DEMAND,
        'seed_uniform_control': SEED_UNIFORM,
        'shortest_paths': 'unweighted hop count, undirected projection, largest component',
    },
    'network': {
        'stops': int(len(lenses)),
        'largest_component_nodes': int(Gc.number_of_nodes()),
        'largest_component_edges': int(Gc.number_of_edges()),
        'largest_component_share': round(Gc.number_of_nodes() / G.number_of_nodes(), 4),
    },
    'proxy_coverage': {
        'cbs_population_total': round(national_population, 0),
        'population_allocated_to_stops': round(allocated, 0),
        'stops_without_cbs_unit': no_unit,
        'stops_whose_unit_has_no_population': no_population,
        'stops_with_zero_demand_weight': int((proxy['demand_weight'] <= 0).sum()),
        'demand_weight_share_of_top_1_percent_of_stops': round(share_top1pct, 4),
        'spearman_population_proxy_vs_service_volume': round(rho_pop_service, 4),
    },
    'agreement': {
        'control_vs_demand_spearman': _agreement(
            'uniform control (same estimator)', 'demand-weighted PROXY',
            'spearman_all_stations'),
        'control_vs_demand_top50_overlap': _agreement(
            'uniform control (same estimator)', 'demand-weighted PROXY', 'top50_overlap'),
        'nb04_vs_demand_spearman': _agreement(
            'topological nb04', 'demand-weighted PROXY', 'spearman_all_stations'),
        'nb04_vs_demand_top50_overlap': _agreement(
            'topological nb04', 'demand-weighted PROXY', 'top50_overlap'),
        'nb04_vs_control_spearman': _agreement(
            'topological nb04', 'uniform control (same estimator)', 'spearman_all_stations'),
        'median_abs_rank_shift_top2000': float(eligible['rank_shift'].abs().median()),
    },
    'biggest_gainer': (None if top_gainer is None else {
        'stop_id': str(top_gainer['stop_id']),
        'stop_name': str(top_gainer['stop_name']),
        'topological_rank': float(top_gainer['topological_rank']),
        'demand_weighted_rank': float(top_gainer['demand_weighted_rank']),
        'rank_shift': float(top_gainer['rank_shift']),
    }),
    'biggest_loser': (None if top_loser is None else {
        'stop_id': str(top_loser['stop_id']),
        'stop_name': str(top_loser['stop_name']),
        'topological_rank': float(top_loser['topological_rank']),
        'demand_weighted_rank': float(top_loser['demand_weighted_rank']),
        'rank_shift': float(top_loser['rank_shift']),
    }),
    'sensitivity_run': {
        'weight_level': bool(RUN_WEIGHT_SENSITIVITY),
        'betweenness_level': bool(RUN_BETWEENNESS_SENSITIVITY),
        'sensitivity_k': SENSITIVITY_K,
    },
    'outputs': sorted(p.name for p in TABLES.glob('*.csv')),
}

with open(STAGE / 'demand_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2)[:2500])
print('\nwritten ->', STAGE / 'demand_summary.json')

## 19. Limitations of the demand proxy - read this before citing any number above

This section is deliberately longer than usual. The whole notebook rests on a proxy, and the proxy is wrong in at least the following ways.

**1. Population near a stop isn't boardings.** This is the biggest assumption. Two stops with identical resident populations may have completely different boarding counts, depending on car ownership, the walking environment, parallel service, fare zones, and habit. Nothing in the GTFS feed can tell them apart. `population_served` is an *exposure* measure - how many people the stop is physically close to - not a usage measure.

**2. Only residential population is counted; destinations are ignored.** Real demand is a matrix of origins and destinations. Workplaces, hospitals, universities, malls, army bases, and beaches generate huge trip volumes and have almost no residents. Our model weights a path's *origin* and its *destination* by the same residential figure, so a stop serving a large employment center with few residents - exactly the kind of place with the highest actual boarding count at the afternoon peak - is systematically underweighted. This is the proxy's most serious structural flaw, and it biases the entire "losers" list: some of those stops aren't overrated by topology at all, they just serve workplaces instead of homes.

**3. No car ownership, income, or age structure.** Public-transport demand per resident varies several-fold between neighborhoods. Because car ownership correlates with socioeconomic cluster, the equity analysis in section 16 is partly circular: low-cluster areas have both higher public-transport dependence *and* higher density, and our proxy captures only the density half.

**4. No time of day.** The proxy is a single static number per stop. Real demand has a morning peak flowing toward employment centers and an evening peak flowing back, and criticality at 08:00 is a different question from criticality at 23:00. Notebooks 19 and 20 handle the time dimension for topology; combining the two - demand-weighted criticality *per time window* - is the natural next step and isn't done here.

**5. The area-to-stop split is naive.** Population is split equally among a statistical area's stops, so two stops on opposite sides of the same street each get half a share even though to the passenger they're one pair of stops. Areas with many stops dilute themselves; areas with one stop concentrate. The gravity kernel softens this but doesn't fix it.

**6. The CBS join is imperfect and inherited.** About a fifth of stops were matched to their statistical area by the *nearest polygon* rather than by containment (see the join-quality print in section 7), and a few hundred stops have no area or no population at all. Those stops have a demand weight of exactly 0, meaning the demand lens treats them as places where nobody starts or ends a trip. For a stop in an unpopulated industrial zone that's roughly right; for a stop whose polygon join simply failed it's wrong.

**7. Shortest paths are measured in hops, and the trip model as a whole is coarse.** Passengers don't travel along shortest paths in the stop-adjacency graph; they travel along *lines*, with transfer penalties, waiting times, and a strong preference not to change vehicles. A path model that ignores transfer cost overcounts paths that zigzag between lines. Notebook 18 builds travel-time edges, which would be a better basis, but mixing that change into the current comparison would make it impossible to attribute any difference to demand weighting.

**8. Both lenses are sampled estimates.** The demand lens uses `K_DEMAND_SOURCES` sampled sources and notebook 04's column used 300. Individual rank shifts of a few hundred places are within the noise, which is exactly why section 15 restricts attention to stations reaching the top 2,000 in at least one lens, and why section 17b reports a noise floor.

**9. Service volume is a partly circular component.** Multiplying by `stop_use_count` imports network structure back into a weight that was supposed to be independent of it. The `pure_population_no_service` sensitivity variant exists precisely so this can be quantified; if the two rankings agree well, the service term isn't what's doing the work, and if they don't - the base demand lens is partly a service ranking in disguise.

**What would fix this:** per-stop boarding counts from the operators' automatic fare-collection data (Rav-Kav), an origin-destination matrix from cellular or survey data, and employee counts per statistical area. All three exist in Israel; none of them is in a public GTFS feed.

## 20. Conclusions

Read these together with the tables in `outputs/nb/21_demand_weighted_criticality/tables/`. The exact numbers depend on the feed snapshot, the proxy parameters, and the sampling seeds.

1. **Demand weighting can be applied to this data, but only as a proxy.** GTFS has no ridership data. What we built is residential population near a stop, gravity-smoothed and preserved at the national level, and optionally multiplied by scheduled service. Every item this notebook writes says so in its column names, and `demand_summary.json` carries `proxy_is_not_ridership: true`.

2. **The two rankings agree at the very top and diverge below it.** The national bridging stations - the ones lying on shortest paths from almost everywhere - stay critical under any weighting, because they're critical for structural reasons that have nothing to do with who lives nearby. The top-50 and top-200 overlap figures in `rank_agreement.csv` are the honest measure of how much the picture changes; the overall Spearman correlation is inflated by the thousands of low-score stations that never move, and shouldn't be cited on its own.

3. **The result is a list of names, not a coefficient.** `rank_movers.csv` is the product. The gainers are usually low-degree stops embedded in dense residential neighborhoods - structurally unremarkable, but many people live around them. The losers are usually nodes on intercity corridors and stops in sparsely populated areas: topologically central, demographically empty. Whether this reweighting is *right* depends entirely on whether you think criticality means "the network breaks" or "people get stranded". This notebook doesn't decide that; it shows that the two definitions give different answers below the top layer.

4. **The equity signal in section 16 is suggestive at best and partly mechanical.** Any promotion of low-cluster neighborhoods comes partly from the fact that our proxy *is* population density, and dense areas in Israel tend toward lower clusters. This deserves to be reported as a hypothesis - a topological criticality ranking may underserve dense poor neighborhoods - and not as a measured finding.

5. **The single biggest flaw is the absence of destinations.** Weighting both endpoints by residential population means employment centers, hospitals, and universities are invisible as trip attractors. Any operational use of this ranking would need an origin-destination matrix; without one, the "losers" list in particular should be treated as a list of *questions*, not conclusions.

6. **The next step.** Combine this weighting with the time-of-day graphs of notebooks 19-20 and the travel-time edges of notebook 18: demand-weighted criticality at the morning peak, on a network where distance is measured in minutes, is the best approximation a GTFS-only study can reach to the question an operator actually asks.